In [1]:
library(tidyverse)
library(tsibble)
library(fable)
library(fabletools)
library(feasts)
library(lubridate)

options(repr.plot.width = 12, repr.plot.height = 5, repr.plot.res = 150)

Warning message:
"package 'tidyverse' was built under R version 4.5.3"
Warning message:
"package 'ggplot2' was built under R version 4.5.3"
Warning message:
"package 'tibble' was built under R version 4.5.3"
Warning message:
"package 'tidyr' was built under R version 4.5.3"
Warning message:
"package 'readr' was built under R version 4.5.3"
Warning message:
"package 'purrr' was built under R version 4.5.3"
Warning message:
"package 'dplyr' was built under R version 4.5.3"
Warning message:
"package 'stringr' was built under R version 4.5.2"
Warning message:
"package 'forcats' was built under R version 4.5.2"
Warning message:
"package 'lubridate' was built under R version 4.5.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ──────────────────────────────────────

In [2]:
# Load pre-cleaned data (produced by data_cleaning.ipynb)
df_lagged  <- readr::read_rds("fluview_clean/df_lagged.rds")
train_full <- readr::read_rds("fluview_clean/train_full.rds")
holdout    <- readr::read_rds("fluview_clean/holdout.rds")
n_train_init <- nrow(train_full)

cat("Rows:", nrow(df_lagged), "\n")
cat("Holdout range:", as.character(min(holdout$week_start)),
    "to", as.character(max(holdout$week_start)), "\n")

Rows: 1198 
Holdout range: 2023 W01 to 2024 W52 


In [3]:
rolling_rmse <- function(data, model_expr, model_name, min_train = n_train_init) {

  stretched <- data %>%
    stretch_tsibble(.init = min_train, .step = 1)

  rolling_fits <- rlang::inject(
    model(stretched, mod = !!model_expr)
  )

  rolling_fc <- rolling_fits %>%
    forecast(h = 1)

  actuals <- data %>%
    as_tibble() %>%
    select(week_start, actual = percent_weighted_ili)

  results <- rolling_fc %>%
    as_tibble() %>%
    left_join(actuals, by = "week_start") %>%
    mutate(
      model    = model_name,
      forecast = .mean,
      sq_error = (actual - forecast)^2
    ) %>%
    select(model, week_start, actual, forecast, sq_error)

  results
}

In [4]:
cat("Fitting rolling MEAN baseline...\n")
res_mean <- rolling_rmse(
  df_lagged,
  expr(MEAN(percent_weighted_ili)),
  "Mean"
)

cat("Fitting rolling SNAIVE baseline...\n")
res_snaive <- rolling_rmse(
  df_lagged,
  expr(SNAIVE(percent_weighted_ili ~ lag(52))),
  "SNaive"
)

cat("Fitting rolling STL+ETS...\n")
res_ets <- rolling_rmse(
  df_lagged,
  expr(
    decomposition_model(
      STL(percent_weighted_ili ~ season(period = 52)),
      ETS(season_adjust ~ error("A") + trend("Ad")))
  ),
  "STL+ETS"
)

Fitting rolling MEAN baseline...


Fitting rolling SNAIVE baseline...
Fitting rolling STL+ETS...


Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."
Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."
Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."
Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."
Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."
Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."
Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS. Seasonality will be ignored."
Warning message:
"Seasonal periods (`period`) of length greather than 24 are not supported by
ETS

In [5]:
all_results <- bind_rows(
  res_mean, res_snaive, res_ets
) %>%
  filter(week_start %in% holdout$week_start)

rmse_table <- all_results %>%
  group_by(model) %>%
  summarise(
    RMSE = sqrt(mean(sq_error, na.rm = TRUE)),
    n    = sum(!is.na(sq_error)),
    .groups = "drop"
  ) %>%
  arrange(RMSE) %>%
  mutate(RMSE = round(RMSE, 4))

print(rmse_table)

# A tibble: 3 × 3
  model    RMSE     n
  <chr>   <dbl> <int>
1 STL+ETS 0.304   104
2 SNaive  1.01    104
3 Mean    1.40    104
